<a href="https://colab.research.google.com/github/vivaan3141/UC-Dashboard-Construction-Vivaan-Gupta/blob/main/UC_Question_Sprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Problem 1:

In [15]:
import pandas as pd

# Load dataset
try:
    df_eth = pd.read_csv('uc_admissions_summary_by_ethnicity.csv')
except FileNotFoundError:
    df_eth = pd.read_csv('Data/uc_admissions_summary_by_ethnicity.csv')

# Ensure 'n' is numeric
if df_eth['n'].dtype == object:
    df_eth['n'] = df_eth['n'].astype(str).str.replace(',', '').astype(float)

# Filter for Fall 2025 Freshman Applicants
fall_2025_apps = df_eth[
    (df_eth['fall_term'] == 2025) &
    (df_eth['entrant_level'].str.lower() == 'freshman') &
    (df_eth['count_type'] == 'App')
]

# Total individual campus applications (excluding Systemwide)
campus_apps = fall_2025_apps[fall_2025_apps['campus'] != 'Systemwide']['n'].sum()

# Total unique applicants (Systemwide total)
unique_applicants = fall_2025_apps[fall_2025_apps['campus'] == 'Systemwide']['n'].sum()

# Calculate average rounded to two decimal places
avg_campuses = round(campus_apps / unique_applicants, 2)

print("Total Individual Campus Applications:", campus_apps)
print("Total Unique Applicants (Systemwide):", unique_applicants)
print("Question 1 Final Answer:", avg_campuses)

Total Individual Campus Applications: 932623
Total Unique Applicants (Systemwide): 205389
Question 1 Final Answer: 4.54


Problem 2

In [2]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('dashboard_data.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/dashboard_data.csv')

# Clean numeric columns if formatted as strings
for col in ['applicants', 'admits']:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Filter for:
# 1. Fall 2025
# 2. Campus is UCLA (Los Angeles)
# 3. School type is Public (California public high schools)
ucla_ca_pub = df[
    (df['fall_term'] == 2025) &
    (df['campus'].str.contains('Los Angeles', case=False, na=False)) &
    (df['school_type'].str.contains('Public', case=False, na=False))
]

total_admits = ucla_ca_pub['admits'].sum()
total_applicants = ucla_ca_pub['applicants'].sum()

admit_rate = total_admits / total_applicants

print(f"Total CA Public Applicants to UCLA (2025): {total_applicants}")
print(f"Total CA Public Admits to UCLA (2025): {total_admits}")
print(f"Admit Rate (Decimal): {admit_rate:.4f}")
print(f"Admit Rate (Percentage): {admit_rate * 100:.2f}%")

Total CA Public Applicants to UCLA (2025): 17965.0
Total CA Public Admits to UCLA (2025): 1487.0
Admit Rate (Decimal): 0.0828
Admit Rate (Percentage): 8.28%


Problem 3

In [3]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('uc_freshman_admission_by_discipline.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/uc_freshman_admission_by_discipline.csv')

# Clean numeric columns if they are formatted as strings
for col in ['applicants', 'admits', 'enrollees']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Inspect column names to be dynamic
year_col = [c for c in df.columns if 'year' in c.lower() or 'term' in c.lower()][0]
campus_col = [c for c in df.columns if 'campus' in c.lower()][0]
disc_col = [c for c in df.columns if 'disc' in c.lower() or 'major' in c.lower()][0]

# Filter for Fall 2025 and exclude systemwide totals
df_2025 = df[(df[year_col] == 2025) & (~df[campus_col].str.contains('systemwide|universitywide', case=False, na=False))]

# 1. Calculate overall admit rate per campus
overall = df_2025.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())

# 2. Calculate Computer Science admit rate per campus
cs_df = df_2025[df_2025[disc_col].str.contains('Computer Science', case=False, na=False)]
cs_rates = cs_df.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())

# 3. Calculate the difference (Overall Rate - CS Rate)
rate_penalty = (overall - cs_rates).dropna().sort_values(ascending=False)

print("--- ADMIT RATE DIFFERENCE (Overall - CS) BY CAMPUS ---")
for campus, diff in rate_penalty.items():
    print(f"{campus}: Overall {overall[campus]:.2%} vs CS {cs_rates[campus]:.2%} (Cost: -{diff*100:.2f}%)")

print("\nQuestion 3 Answer (Campus):", rate_penalty.index[0])

--- ADMIT RATE DIFFERENCE (Overall - CS) BY CAMPUS ---
Davis: Overall 44.30% vs CS 19.33% (Cost: -24.97%)
San Diego: Overall 28.13% vs CS 19.87% (Cost: -8.26%)
Riverside: Overall 86.52% vs CS 81.12% (Cost: -5.41%)
Berkeley: Overall 11.32% vs CS 6.45% (Cost: -4.87%)
Santa Barbara: Overall 38.21% vs CS 33.91% (Cost: -4.29%)
Los Angeles: Overall 9.59% vs CS 7.32% (Cost: -2.27%)
Irvine: Overall 29.35% vs CS 27.58% (Cost: -1.77%)
Santa Cruz: Overall 72.51% vs CS 79.39% (Cost: --6.87%)

Question 3 Answer (Campus): Davis


/tmp/ipykernel_2711/641274511.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall = df_2025.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())
/tmp/ipykernel_2711/641274511.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cs_rates = cs_df.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())


Problem 4


In [4]:
import pandas as pd

# Load dataset
try:
    df_disc = pd.read_csv('uc_freshman_admission_by_discipline.csv')
except FileNotFoundError:
    df_disc = pd.read_csv('Data/uc_freshman_admission_by_discipline.csv')

# Identify relevant columns dynamically
year_col = [c for c in df_disc.columns if 'year' in c.lower() or 'term' in c.lower()][0]
campus_col = [c for c in df_disc.columns if 'campus' in c.lower()][0]
disc_col = [c for c in df_disc.columns if 'disc' in c.lower() or 'major' in c.lower()][0]

# Filter for UC Berkeley, Fall 2025, Computer Science
cal_cs = df_disc[
    (df_disc[campus_col].str.contains('Berkeley', case=False, na=False)) &
    (df_disc[year_col] == 2025) &
    (df_disc[disc_col].str.contains('Computer Science', case=False, na=False))
]

# Find 75th and 25th admit GPA percentiles
p75_col = [c for c in df_disc.columns if '75' in c and ('admit' in c.lower() or 'adm' in c.lower())][0]
p25_col = [c for c in df_disc.columns if '25' in c and ('admit' in c.lower() or 'adm' in c.lower())][0]

p75 = float(cal_cs[p75_col].values[0])
p25 = float(cal_cs[p25_col].values[0])

# Interquartile Range (IQR) = Q3 - Q1
iqr = round(p75 - p25, 2)

print(f"25th Percentile Admit GPA (Q1): {p25}")
print(f"75th Percentile Admit GPA (Q3): {p75}")
print(f"Question 4 Answer (IQR): {iqr}")

25th Percentile Admit GPA (Q1): 4.2
75th Percentile Admit GPA (Q3): 4.29
Question 4 Answer (IQR): 0.09


Problem 5

In [5]:
import pandas as pd

# Load dataset
try:
    df_eth = pd.read_csv('uc_admissions_summary_by_ethnicity.csv')
except FileNotFoundError:
    df_eth = pd.read_csv('Data/uc_admissions_summary_by_ethnicity.csv')

# Ensure 'n' is numeric
if df_eth['n'].dtype == object:
    df_eth['n'] = df_eth['n'].astype(str).str.replace(',', '').astype(float)

# Filter for Fall 2025 Freshman data excluding Systemwide totals
df_2025 = df_eth[
    (df_eth['fall_term'] == 2025) &
    (df_eth['entrant_level'].str.lower() == 'freshman') &
    (~df_eth['campus'].str.contains('systemwide|universitywide', case=False, na=False)) &
    (df_eth['count_type'].isin(['App', 'Adm']))
]

# Pivot table to get counts by Campus, Ethnicity, and Count Type (App vs Adm)
pivot = df_2025.pivot_table(
    index=['campus', 'ethnicity'],
    columns='count_type',
    values='n',
    aggfunc='sum'
).reset_index()

pivot['admit_rate'] = pivot['Adm'] / pivot['App']

# Extract rates for White and Hispanic/Latino(a)
white_rates = pivot[pivot['ethnicity'].str.contains('White', case=False, na=False)].set_index('campus')['admit_rate']
hisp_rates = pivot[pivot['ethnicity'].str.contains('Hispanic', case=False, na=False)].set_index('campus')['admit_rate']

# Compare campus by campus
comparison_df = pd.DataFrame({
    'White_Admit_Rate': white_rates,
    'Hispanic_Admit_Rate': hisp_rates
})
comparison_df['White_Higher'] = comparison_df['White_Admit_Rate'] > comparison_df['Hispanic_Admit_Rate']

print("--- CAMPUS-BY-CAMPUS ADMIT RATES (FALL 2025) ---")
print(comparison_df.applymap(lambda x: f"{x:.2%}" if isinstance(x, float) else x))

count_higher = int(comparison_df['White_Higher'].sum())
print(f"\nQuestion 5 Answer: {count_higher}")

--- CAMPUS-BY-CAMPUS ADMIT RATES (FALL 2025) ---
              White_Admit_Rate Hispanic_Admit_Rate  White_Higher
campus                                                          
Berkeley                12.02%              11.81%          True
Davis                   45.04%              35.87%          True
Irvine                  27.48%              18.63%          True
Los Angeles             10.00%               7.53%          True
Merced                  96.91%              95.08%          True
Riverside               90.23%              83.27%          True
San Diego               27.94%              25.91%          True
Santa Barbara           38.19%              30.85%          True
Santa Cruz              79.10%              61.92%          True

Question 5 Answer: 9


/tmp/ipykernel_2711/1204549388.py:43: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(comparison_df.applymap(lambda x: f"{x:.2%}" if isinstance(x, float) else x))


Problem 6

In [6]:
import pandas as pd

# Load dataset
try:
    df_eth = pd.read_csv('uc_admissions_summary_by_ethnicity.csv')
except FileNotFoundError:
    df_eth = pd.read_csv('Data/uc_admissions_summary_by_ethnicity.csv')

# Ensure 'n' is numeric
if df_eth['n'].dtype == object:
    df_eth['n'] = df_eth['n'].astype(str).str.replace(',', '').astype(float)

# Filter for Fall 2025 Freshman Systemwide totals
df_sys = df_eth[
    (df_eth['fall_term'] == 2025) &
    (df_eth['entrant_level'].str.lower() == 'freshman') &
    (df_eth['campus'].str.contains('systemwide', case=False, na=False)) &
    (df_eth['count_type'].isin(['App', 'Adm']))
]

# Pivot table for White vs Hispanic/Latino(a)
pivot = df_sys.pivot_table(
    index='ethnicity',
    columns='count_type',
    values='n',
    aggfunc='sum'
).reset_index()

pivot['admit_rate'] = pivot['Adm'] / pivot['App']

# Extract rates
white_row = pivot[pivot['ethnicity'].str.contains('White', case=False, na=False)]
hisp_row = pivot[pivot['ethnicity'].str.contains('Hispanic', case=False, na=False)]

white_rate = white_row['admit_rate'].values[0]
hisp_rate = hisp_row['admit_rate'].values[0]

print(f"White Systemwide Admit Rate (2025): {white_rate:.4f} ({white_rate:.2%})")
print(f"Hispanic/Latino(a) Systemwide Admit Rate (2025): {hisp_rate:.4f} ({hisp_rate:.2%})")

higher_group = "White" if white_rate > hisp_rate else "Hispanic/Latino(a)"
print(f"\nQuestion 6 Answer: {higher_group}")

White Systemwide Admit Rate (2025): 0.6868 (68.68%)
Hispanic/Latino(a) Systemwide Admit Rate (2025): 0.7453 (74.53%)

Question 6 Answer: Hispanic/Latino(a)


Problem 7

In [7]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('dashboard_data.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/dashboard_data.csv')

# Filter for Class of 2023 graduates in the Bay Area
# (In dashboard_data.csv, fall_term corresponds to the cohort/grad year, and county defines Bay Area)
bay_counties = [
    'Alameda', 'Contra Costa', 'Marin', 'Napa', 'San Francisco',
    'San Mateo', 'Santa Clara', 'Solano', 'Sonoma'
]

# Ensure numeric columns
for col in ['graduates', 'enrolled_ccc']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Filter unique school records to avoid duplicate counts across campus rows
# Using campus == 'Universitywide' or dropping duplicates by high_school/cds_code
df_2023 = df[
    (df['fall_term'] == 2023) &
    (df['county'].isin(bay_counties)) &
    (df['campus'] == 'Universitywide')
].copy()

total_grads = df_2023['graduates'].sum()
total_ccc = df_2023['enrolled_ccc'].sum()

ccc_share = total_ccc / total_grads

print(f"Total Bay Area Graduates (Class of 2023): {total_grads:.0f}")
print(f"Total Enrolled in CCC within 12 Months: {total_ccc:.0f}")
print(f"CCC Share (Decimal): {ccc_share:.4f}")
print(f"CCC Share (Percentage): {ccc_share * 100:.2f}%")
print(f"\nQuestion 7 Answer: {ccc_share:.4f}")

Total Bay Area Graduates (Class of 2023): 63578
Total Enrolled in CCC within 12 Months: 21644
CCC Share (Decimal): 0.3404
CCC Share (Percentage): 34.04%

Question 7 Answer: 0.3404


Problem 8

In [8]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('dashboard_data.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/dashboard_data.csv')

# Ensure numeric columns
for col in ['applicants', 'ag_completers']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Filter for:
# 1. Mission San Jose High School
# 2. Fall 2023
# 3. Universitywide row (applicants who applied to at least one UC)
msj_2023 = df[
    (df['high_school'].str.contains('Mission San Jose', case=False, na=False)) &
    (df['fall_term'] == 2023) &
    (df['campus'] == 'Universitywide')
]

uc_applicants = float(msj_2023['applicants'].values[0])
ag_completers = float(msj_2023['ag_completers'].values[0])

share = uc_applicants / ag_completers

print(f"Mission San Jose HS (Fall 2023):")
print(f"Universitywide Applicants: {uc_applicants:.0f}")
print(f"a-g Completers: {ag_completers:.0f}")
print(f"Share (Decimal): {share:.4f}")
print(f"Share (Percentage): {share * 100:.2f}%")
print(f"\nQuestion 8 Answer: {share:.4f}")

Mission San Jose HS (Fall 2023):
Universitywide Applicants: 420
a-g Completers: 424
Share (Decimal): 0.9906
Share (Percentage): 99.06%

Question 8 Answer: 0.9906


Problem 9

In [9]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('dashboard_data.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/dashboard_data.csv')

# Ensure 'applicants' is numeric
if df['applicants'].dtype == object:
    df['applicants'] = df['applicants'].astype(str).str.replace(',', '').astype(float)

# Filter for:
# 1. Fall 2025
# 2. California Public High Schools
# 3. At least one applicant (applicants >= 1)
pub_2025 = df[
    (df['fall_term'] == 2025) &
    (df['school_type'].str.contains('Public', case=False, na=False)) &
    (df['applicants'] >= 1)
]

# Count unique schools by high_school name (and cds_code if available)
distinct_schools_name = pub_2025['high_school'].nunique()
distinct_schools_cds = pub_2025['cds_code'].dropna().nunique()

print(f"Distinct CA Public High Schools by Name: {distinct_schools_name}")
print(f"Distinct CA Public High Schools by CDS Code: {distinct_schools_cds}")
print(f"\nQuestion 9 Answer: {distinct_schools_name}")

Distinct CA Public High Schools by Name: 209
Distinct CA Public High Schools by CDS Code: 212

Question 9 Answer: 209


Problem 10

In [10]:
import pandas as pd
import statsmodels.api as sm

# Load dataset
try:
    df = pd.read_csv('dashboard_data.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/dashboard_data.csv')

# The 5 target schools from the form
target_schools = [
    'HERCULES HIGH SCHOOL',
    'MISSION SENIOR HIGH SCHOOL',
    'MONTEREY TRAIL HIGH SCHOOL',
    'PHILLIP & SALA BURTON ACAD HS',
    'RANCHO SAN JUAN HIGH SCHOOL'
]

# Check if precomputed residuals already exist in the dataset
cal_data = df[
    (df['campus'].str.contains('Berkeley', case=False, na=False)) &
    (df['fall_term'].between(2022, 2025))
].copy()

# Option A: If precomputed residual columns exist
if 'admit_rate_residual' in cal_data.columns and cal_data['admit_rate_residual'].notna().sum() > 0:
    target_df = cal_data[cal_data['high_school'].str.upper().isin(target_schools)]
    residual_rank = target_df.groupby('high_school')['admit_rate_residual'].mean().sort_values(ascending=False)
else:
    # Option B: Fit OLS regression controlling for a-g, poverty, GPA, school size
    for col in ['applicants', 'admits', 'ag_completion_rate', 'frpm_pct', 'applicant_gpa', 'graduates']:
        if col in cal_data.columns and cal_data[col].dtype == object:
            cal_data[col] = cal_data[col].astype(str).str.replace(',', '').astype(float)

    cal_data['admit_rate'] = cal_data['admits'] / cal_data['applicants']

    # Feature columns matching dataset schema
    features = ['ag_completion_rate', 'frpm_pct', 'applicant_gpa', 'graduates']
    model_df = cal_data.dropna(subset=features + ['admit_rate']).copy()

    X = sm.add_constant(model_df[features])
    y = model_df['admit_rate']

    model = sm.OLS(y, X).fit()
    model_df['residual'] = model.resid

    target_df = model_df[model_df['high_school'].str.upper().isin(target_schools)]
    residual_rank = target_df.groupby('high_school')['residual'].mean().sort_values(ascending=False)

print("--- AVERAGE RESIDUAL ADMIT RATE (2022-2025) ---")
for school, res in residual_rank.items():
    print(f"{school}: {res:+.4f}")

print(f"\nQuestion 10 Answer: {residual_rank.index[0]}")

--- AVERAGE RESIDUAL ADMIT RATE (2022-2025) ---
MISSION SENIOR HIGH SCHOOL: +0.2500
HERCULES HIGH SCHOOL: +0.0755
PHILLIP & SALA BURTON ACAD HS: -0.1064

Question 10 Answer: MISSION SENIOR HIGH SCHOOL
